# W07 -- Data Summary: Target Definition, Feature Engineering, and Modeling

**Two more refinements on top of the previous revision** (which switched the target from `is_unaffordable` to `collapse_onset` and rebuilt the training population around real events):

1. **Confirmed onsets only.** Investigating two leave-one-city-out backtest failures (Springfield, MA and Traverse City, MI, both AUC 0.0) found that their only `collapse_onset` event crossed the 5.0 threshold by a razor-thin margin with zero confirmed quarters afterward -- the label itself was unverifiable. This turned out to generalize: 26 of 168 raw onset events revert below the threshold the very next quarter (most because we can observe the reversion directly, a few because the data window ends right after). A new `collapse_onset_confirmed` label requires the metro to still be unaffordable the following quarter, which rules out both single-quarter statistical noise and unverifiable right-censored events in one rule.
2. **Near-miss cities added as hard negatives.** 33 additional cities reached a price-to-income ratio of 4.5-5.0 at some point but never crossed 5.0. Their data is added to the training population as informative "got close but didn't collapse" negative examples, on top of the 54 cities with a confirmed real event.

**Input:** `data/final_data/price_changes_with_collapse_flags.csv`.

**Outputs:** train/val/holdout CSV splits, metrics tables, and SHAP/risk-ranking figures under `output/`.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import optuna

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold, GroupKFold,
    cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    make_scorer, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.discrete.discrete_model import Logit

# Data assembly, labelling, and feature engineering all live in the
# housing_pipeline package now, so this notebook is purely about modeling.
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
from housing_pipeline import (
    FEATURE_NAMES,
    MONOTONE_CONSTRAINTS,
    base_rates,
    build_features,
    build_watchlist,
    lead_time,
    lead_time_summary,
    load_panel,
    walk_forward,
    watchlist_summary,
)

In [2]:
# The panel is built once by the pipeline, not by this notebook:
#
#     python -m housing_pipeline build
#
# It arrives already joined across all nine sources and already labelled, so
# nothing here re-derives data. Rebuild it when upstream data refreshes.
df = load_panel()

print(f"panel: {len(df):,} rows, {df['cbsa'].nunique()} metros, "
      f"{df['year'].min()}-{df['year'].max()}")
print(f"confirmed onsets: {int(df['collapse_onset_confirmed'].sum())}")

panel: 71,072 rows, 410 metros, 1975-2026
confirmed onsets: 124


## Target definition: confirmed affordability collapse onset

The three labels below are computed by the pipeline (`housing_pipeline.panel.add_affordability_labels`) rather than here, so the modeling notebook and any other consumer cannot drift apart in how they define the target:

* `is_unaffordable` — price-to-income above 5.0. A persistent *state*, kept for reference only: it changes about 4% of the time year over year, so a "nothing changed" baseline beats a trained model on it.
* `collapse_onset` — the first quarter a metro crosses the threshold. A real transition event.
* **`collapse_onset_confirmed`** — an onset that still holds the following quarter. This is the modeling target. It excludes single-quarter reversions *and* crossings at the edge of the data window that cannot be verified yet.

Modeling is further restricted to **at-risk rows** (`prev_unaffordable == False`): once a metro is already unaffordable, predicting an onset for it means nothing.

In [3]:
# Austin (12420), Boise (14260), Tampa (45294) are the original case studies.
# Their onset dates are a sanity check that the join upstream is intact:
# Austin 2021Q2, Boise 2019Q3, Tampa 2021Q4.
ANCHOR_CITIES = {'Austin': 12420, 'Boise': 14260, 'Tampa': 45294}

confirmed = (
    df[df['collapse_onset_confirmed']]
    .groupby('cbsa')[['metro_name', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
)
print("Confirmed collapse onset (anchor cities):")
print(confirmed.loc[confirmed.index.isin(ANCHOR_CITIES.values())])

print("\nRaw onsets:", int(df['collapse_onset'].sum()),
      "| confirmed:", int(df['collapse_onset_confirmed'].sum()),
      "| dropped as unconfirmed:",
      int(df['collapse_onset'].sum() - df['collapse_onset_confirmed'].sum()))

Confirmed collapse onset (anchor cities):
                             metro_name  year  qtr  price_to_income_ratio
cbsa                                                                     
12420  Austin-Round Rock-San Marcos, TX  2021    2               5.269679
14260                    Boise City, ID  2019    1               5.012043
45294                  Tampa, FL (MSAD)  2021    3               5.022391

Raw onsets: 134 | confirmed: 124 | dropped as unconfirmed: 10


In [4]:
# Sensitivity check: does the onset date move much if the threshold isn't 5.0?
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code_ in ANCHOR_CITIES.items():
        sub = df[df['cbsa'] == code_].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Result: 5.0 gives the tightest, most realistic cluster of onset dates.


--- Threshold: 4.0 ---
  Austin: 2014Q3
  Boise: 2016Q1
  Tampa: 2017Q3

--- Threshold: 4.5 ---
  Austin: 2021Q1
  Boise: 2017Q4
  Tampa: 2021Q1

--- Threshold: 5.0 ---
  Austin: 2021Q2
  Boise: 2019Q1
  Tampa: 2021Q3

--- Threshold: 5.5 ---
  Austin: 2021Q3
  Boise: 2020Q4
  Tampa: 2022Q2


## Feature engineering

Feature construction lives in `housing_pipeline.features`, which is unit-tested for the property that matters most here: **no feature may observe the quarter that defines its own label.** Every predictor is lagged four quarters, and the lag is applied *after* any percent-change or rolling calculation.

Each feature also declares whether its direction is knowable. Features with an unambiguous relationship to risk carry a monotonic constraint into the model; features whose sign is genuinely arguable (unemployment, inventory, the S&P 500 return, and both wage/employment series) are left unconstrained rather than having a direction imposed on them.

In [5]:
df = build_features(df)

ALL_FEATURES = FEATURE_NAMES
print(f"{len(ALL_FEATURES)} features engineered:")
for name in ALL_FEATURES:
    metros = df.loc[df[name].notna(), 'cbsa'].nunique()
    print(f"  {name:<38} {metros:>4} metros")

15 features engineered:
  price_to_income_lag                     364 metros
  price_to_income_5yr_chg                 363 metros
  zhvi_yoy_lag                            371 metros
  zhvi_qoq_lag                            371 metros
  three-year_home_price_growth_trend      371 metros
  hpi_yoy_lag                             410 metros
  hpi_3yr_chg_lag                         410 metros
  pop_velocity_lag                        410 metros
  pop_acceleration_lag                    410 metros
  zori_yoy_lag                            367 metros
  unemployment_rate_lag                   373 metros
  inv_qoq_lag                             373 metros
  sp500_yoy_lag                           410 metros
  qcew_wage_yoy_lag                       373 metros
  qcew_emp_yoy_lag                        373 metros


## Selecting the training population: confirmed events + near-miss cities

**Event cities:** every city with complete feature data that has at least one *confirmed* `collapse_onset_confirmed` event.

**Near-miss cities (new):** cities that reached a price-to-income ratio of 4.5-5.0 at some point but never crossed 5.0. These contribute only negative examples, but they're a qualitatively different, more informative negative than a permanently-affordable city -- "got close and didn't collapse" is a harder, more useful example for the model to learn from than "was never remotely close."

In [6]:
at_risk = df[~df["prev_unaffordable"]].dropna(subset=ALL_FEATURES + ["collapse_onset_confirmed"]).copy()
event_cities = at_risk.loc[at_risk["collapse_onset_confirmed"], "cbsa"].unique()

print("At-risk rows with complete features:", len(at_risk))
print("Event cities (>=1 confirmed onset, complete features):", len(event_cities))
print("Confirmed collapse_onset_confirmed=True rows:", int(at_risk["collapse_onset_confirmed"].sum()))

# Near-miss cities: reached 4.5-5.0 but never crossed 5.0, anywhere in their history
near_miss_info = df.groupby("cbsa").agg(
    max_pti=("price_to_income_ratio", "max"),
    ever_onset=("collapse_onset", "any"),
).reset_index()
near_miss_cbsas = near_miss_info[
    (near_miss_info["max_pti"] >= 4.5) & (near_miss_info["max_pti"] < 5.0) & (~near_miss_info["ever_onset"])
]["cbsa"]

near_miss_pool = at_risk[at_risk["cbsa"].isin(near_miss_cbsas)]
near_miss_cities = near_miss_pool["cbsa"].unique()
print(f"Near-miss cities (4.5-5.0, never crossed, complete features): {len(near_miss_cities)}")

combined_cities = set(event_cities) | set(near_miss_cities)
training_cbsa_map = (
    at_risk[at_risk["cbsa"].isin(combined_cities)]
    .groupby("cbsa")["metro_name"].first()
    .reset_index()
    .set_index("metro_name")["cbsa"]
    .to_dict()
)
print(f"\nTotal training cities: {len(training_cbsa_map)} "
      f"({len(event_cities)} event cities + {len(near_miss_cities)} near-miss)")
print("Anchor cities included:", all(c in training_cbsa_map.values() for c in ANCHOR_CITIES.values()))

training_pool_all = at_risk[at_risk["cbsa"].isin(training_cbsa_map.values())].copy()
print(f"\nTraining pool: {len(training_pool_all)} rows, "
      f"positive rate {training_pool_all['collapse_onset_confirmed'].mean():.1%}")

At-risk rows with complete features: 5949
Event cities (>=1 confirmed onset, complete features): 52
Confirmed collapse_onset_confirmed=True rows: 52
Near-miss cities (4.5-5.0, never crossed, complete features): 32

Total training cities: 84 (52 event cities + 32 near-miss)
Anchor cities included: False

Training pool: 1455 rows, positive rate 3.6%


## Findings: why this framing, and what the honest baselines are

- **`is_unaffordable` (the original target) is dominated by persistence** -- it only flips 3.9% of the time over any 4-quarter window, so a trivial "was it already true a year ago" rule beats the fitted model on it.
- **Raw `collapse_onset` includes unverifiable labels** -- 26 of 168 events revert (or can't yet be confirmed) the following quarter.
- **`collapse_onset_confirmed`, restricted to at-risk rows, is the honest target used below.** An "always predict no collapse" baseline scores F1 = 0.0 on it; nothing here can be gamed by persistence or an unconfirmed label.

In [7]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

target = "collapse_onset_confirmed"
model1_pool = training_pool_all.copy()

X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

model1_pool.to_csv("output/model1_pool.csv", index=False)
print("Model 1 pool:", len(model1_pool), "rows,", groups_m1.nunique(), "cities,",
      f"{y_all.mean():.1%} positive")

Model 1 pool: 1455 rows, 84 cities, 3.6% positive


In [8]:
# Model 2 uses the same population as Model 1 -- see the Modeling section for
# how the two differ procedurally.
model2_train_cities = dict(training_cbsa_map)
model2_pool = training_pool_all.copy()

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

model2_pool.to_csv("output/model2_pool.csv", index=False)
print("Model 2 pool:", len(model2_pool), "rows,", groups_m2.nunique(), "cities,",
      f"{yb_all.mean():.1%} positive")

Model 2 pool: 1455 rows, 84 cities, 3.6% positive


In [9]:
# Holdout score set: at-risk metros NOT in the training population, complete
# features. These are the metros actually being ranked for early-warning risk.
holdout_scoring = at_risk[~at_risk["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=ALL_FEATURES
).copy()

print("Holdout scoring rows:", len(holdout_scoring), "| cities:", holdout_scoring["cbsa"].nunique())

holdout_scoring.to_csv("output/holdout_scoring.csv", index=False)
print("Saved: model1_pool.csv, model2_pool.csv, holdout_scoring.csv")

Holdout scoring rows: 4494 | cities: 262
Saved: model1_pool.csv, model2_pool.csv, holdout_scoring.csv


## Modeling: XGBoost + SHAP explainability

With a ~4.7% positive rate (more imbalanced than the event-cities-only population, since near-miss cities add pure-negative rows), both models use `scale_pos_weight`. **PR-AUC (average precision) is the primary reported metric**, always shown next to the no-skill baseline.

In [10]:
os.makedirs("output/figures", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

# Feature list and monotonic constraints are defined once, in
# housing_pipeline.features, and imported here so the model and the pipeline
# cannot disagree about either their identity or their order.
ALL_FEATURES = FEATURE_NAMES
MONOTONE_INCREASING = MONOTONE_CONSTRAINTS

assert len(MONOTONE_INCREASING) == len(ALL_FEATURES)
print(f"{len(ALL_FEATURES)} features; "
      f"{sum(1 for c in MONOTONE_INCREASING if c == 0)} left unconstrained")

15 features; 5 left unconstrained


In [11]:
# Reload from the saved splits so this modeling section can be re-run independently
# of the feature-engineering cells above.
model1_pool = pd.read_csv("output/model1_pool.csv")
model2_pool = pd.read_csv("output/model2_pool.csv")
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")

target = "collapse_onset_confirmed"
X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

### Model 1: Early-Warning Indicator Model

Model 1 identifies which of the engineered indicators is most associated with a *confirmed* `collapse_onset_confirmed` event, across every city with complete feature data that has ever had one, plus the near-miss cities. Default hyperparameters, used for SHAP explainability.

In [12]:
# Grouped split (StratifiedGroupKFold): no city appears on both sides.
neg1, pos1 = (y_all == 0).sum(), (y_all == 1).sum()
scale_pos_weight_1 = neg1 / pos1
print(f"Model 1 class balance: neg={neg1}, pos={pos1}, scale_pos_weight={scale_pos_weight_1:.1f}")

split_m1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_m1, val_idx_m1 = next(split_m1.split(X_all, y_all, groups=groups_m1))
Xa_train, Xa_val = X_all.iloc[train_idx_m1], X_all.iloc[val_idx_m1]
ya_train, ya_val = y_all.iloc[train_idx_m1], y_all.iloc[val_idx_m1]

train_cities_m1 = set(groups_m1.iloc[train_idx_m1])
val_cities_m1 = set(groups_m1.iloc[val_idx_m1])
print("City overlap between train and val (should be 0):", len(train_cities_m1 & val_cities_m1))
print(f"Train: {len(Xa_train)} rows / {len(train_cities_m1)} cities, "
      f"Val: {len(Xa_val)} rows / {len(val_cities_m1)} cities, "
      f"val positive rate: {ya_val.mean():.1%}")

model1_fresh = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight_1,
    monotone_constraints=MONOTONE_INCREASING,
    random_state=42
)
model1_fresh.fit(Xa_train, ya_train)

pred_fresh = model1_fresh.predict(Xa_val)
proba_fresh = model1_fresh.predict_proba(Xa_val)[:, 1]

baseline_prauc_1 = ya_val.mean()
print("\nFresh Model 1 metrics:")
print({
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh),
    "pr_auc": average_precision_score(ya_val, proba_fresh),
    "pr_auc_baseline (no-skill)": baseline_prauc_1
})
print(classification_report(ya_val, pred_fresh, zero_division=0))

Model 1 class balance: neg=1403, pos=52, scale_pos_weight=27.0


City overlap between train and val (should be 0): 0
Train: 1126 rows / 67 cities, Val: 329 rows / 17 cities, val positive rate: 2.7%



Fresh Model 1 metrics:
{'accuracy': 0.8693009118541033, 'precision': 0.14583333333333334, 'recall': 0.7777777777777778, 'f1': 0.24561403508771928, 'roc_auc': 0.9197916666666666, 'pr_auc': 0.23455456906609298, 'pr_auc_baseline (no-skill)': np.float64(0.02735562310030395)}
              precision    recall  f1-score   support

           0       0.99      0.87      0.93       320
           1       0.15      0.78      0.25         9

    accuracy                           0.87       329
   macro avg       0.57      0.82      0.59       329
weighted avg       0.97      0.87      0.91       329



In [13]:
model1_metrics = pd.DataFrame([{
    "model": "Model 1 (explanatory, target=collapse_onset_confirmed, event+near-miss population)",
    "n_training_cities": groups_m1.nunique(),
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh),
    "pr_auc": average_precision_score(ya_val, proba_fresh),
    "pr_auc_baseline": baseline_prauc_1
}])
model1_metrics.to_csv("output/tables/model1_metrics_final.csv", index=False)
print("Saved output/tables/model1_metrics_final.csv")

Saved output/tables/model1_metrics_final.csv


In [14]:
explainer = shap.TreeExplainer(model1_fresh)
shap_values = explainer.shap_values(Xa_train)

plt.figure()
shap.summary_plot(shap_values, Xa_train, show=False)
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved output/figures/shap_summary_model1.png")

shap_importance = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv("output/tables/shap_importance_model1.csv", index=False)
print(shap_importance)

Saved output/figures/shap_summary_model1.png
                               feature  mean_abs_shap
0                  price_to_income_lag       1.112041
3                         zhvi_qoq_lag       0.691184
10               unemployment_rate_lag       0.492796
4   three-year_home_price_growth_trend       0.469411
12                       sp500_yoy_lag       0.375308
14                    qcew_emp_yoy_lag       0.300408
11                         inv_qoq_lag       0.273126
9                         zori_yoy_lag       0.209138
8                 pop_acceleration_lag       0.193610
13                   qcew_wage_yoy_lag       0.112995
2                         zhvi_yoy_lag       0.105251
5                          hpi_yoy_lag       0.075023
1              price_to_income_5yr_chg       0.029276
6                      hpi_3yr_chg_lag       0.019493
7                     pop_velocity_lag       0.008202


### Model 2: Generalization/Scoring Model

Same training population and full feature set as Model 1 -- confirmed identical by construction, not an accident (see the note after tuning below). Optuna-tuned, including `scale_pos_weight` in the search space; its final fitted version is the one used to score the holdout set.

In [15]:
neg2, pos2 = (yb_all == 0).sum(), (yb_all == 1).sum()
class_ratio_2 = neg2 / pos2
print(f"Model 2 class balance: neg={neg2}, pos={pos2}, class_ratio={class_ratio_2:.1f}")

split_m2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_m2, val_idx_m2 = next(split_m2.split(Xb_all, yb_all, groups=groups_m2))
Xb_train, Xb_val = Xb_all.iloc[train_idx_m2], Xb_all.iloc[val_idx_m2]
yb_train, yb_val = yb_all.iloc[train_idx_m2], yb_all.iloc[val_idx_m2]
groups_train_m2 = groups_m2.iloc[train_idx_m2]

train_cities_m2 = set(groups_m2.iloc[train_idx_m2])
val_cities_m2 = set(groups_m2.iloc[val_idx_m2])
print("City overlap between train and val (should be 0):", len(train_cities_m2 & val_cities_m2))
print(f"Train: {len(Xb_train)} rows / {len(train_cities_m2)} cities, "
      f"Val: {len(Xb_val)} rows / {len(val_cities_m2)} cities, "
      f"val positive rate: {yb_val.mean():.1%}")

Model 2 class balance: neg=1403, pos=52, class_ratio=27.0
City overlap between train and val (should be 0): 0
Train: 1126 rows / 67 cities, Val: 329 rows / 17 cities, val positive rate: 2.7%


In [16]:
# Hyperparameter + threshold tuning for Model 2 (Optuna, 100 trials, grouped CV on PR-AUC).
Path("output/tables").mkdir(parents=True, exist_ok=True)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, class_ratio_2 * 1.5),
        "monotone_constraints": MONOTONE_INCREASING,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(
        xgb.XGBClassifier(**params), Xb_train, yb_train,
        groups=groups_train_m2, scoring="average_precision", cv=cv, n_jobs=-1
    ).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = {
    **study.best_params,
    "monotone_constraints": MONOTONE_INCREASING,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}
print("\nBest parameters:", study.best_params)
print(f"Best cross-validation PR-AUC: {study.best_value:.3f} (baseline: {yb_train.mean():.3f})")

base_model = xgb.XGBClassifier(**best_params)
oof_probabilities = cross_val_predict(
    base_model, Xb_train, yb_train, groups=groups_train_m2, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores = [
    f1_score(yb_train, oof_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"Best OOF F1: {max(f1_scores):.3f}")

model2_final = xgb.XGBClassifier(**best_params)
model2_final.fit(Xb_train, yb_train)

validation_probabilities = model2_final.predict_proba(Xb_val)[:, 1]
validation_predictions = (validation_probabilities >= best_threshold).astype(int)

tuned_metrics = {
    "model": "Model 2 (generalization/scoring, target=collapse_onset_confirmed, event+near-miss population)",
    "n_training_cities": groups_m2.nunique(),
    "features": ", ".join(ALL_FEATURES),
    "threshold": round(best_threshold, 2),
    "accuracy": accuracy_score(yb_val, validation_predictions),
    "precision": precision_score(yb_val, validation_predictions, zero_division=0),
    "recall": recall_score(yb_val, validation_predictions, zero_division=0),
    "f1": f1_score(yb_val, validation_predictions, zero_division=0),
    "roc_auc": roc_auc_score(yb_val, validation_probabilities),
    "pr_auc": average_precision_score(yb_val, validation_probabilities),
    "pr_auc_baseline": yb_val.mean()
}

print("\nTuned Model 2 validation metrics:")
for name, value in tuned_metrics.items():
    print(f"{name}: {value}")
print("\nClassification report:")
print(classification_report(yb_val, validation_predictions, zero_division=0))

[I 2026-09-01 14:55:02,939] A new study created in memory with name: no-name-a153a216-2668-402e-817a-d8d1485cde75


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-01 14:55:04,663] Trial 0 finished with value: 0.3334948400988675 and parameters: {'max_depth': 3, 'min_child_weight': 7, 'learning_rate': 0.07259248719561363, 'n_estimators': 140, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.0349658373387986e-08, 'reg_lambda': 0.6245760287469887, 'scale_pos_weight': 24.726703107748772}. Best is trial 0 with value: 0.3334948400988675.


[I 2026-09-01 14:55:05,918] Trial 1 finished with value: 0.37771803062023845 and parameters: {'max_depth': 4, 'min_child_weight': 1, 'learning_rate': 0.13826189316223855, 'n_estimators': 175, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 3.3300161336615e-07, 'reg_lambda': 5.472429642032189e-06, 'scale_pos_weight': 21.712741844714774}. Best is trial 1 with value: 0.37771803062023845.
[I 2026-09-01 14:55:05,965] Trial 2 finished with value: 0.31802411222861654 and parameters: {'max_depth': 3, 'min_child_weight': 3, 'learning_rate': 0.05243180891902853, 'n_estimators': 71, 'subsample': 0.7168578594140872, 'colsample_bytree': 0.7465447373174766, 'reg_alpha': 6.107319200689796e-05, 'reg_lambda': 0.11656915613247415, 'scale_pos_weight': 8.881354574616026}. Best is trial 1 with value: 0.37771803062023845.
[I 2026-09-01 14:55:06,032] Trial 3 finished with value: 0.2967436664402213 and parameters: {'max_depth': 3, 'min_child_weight': 5, 'learning_rate': 0

[I 2026-09-01 14:55:07,276] Trial 5 finished with value: 0.33900922526719834 and parameters: {'max_depth': 3, 'min_child_weight': 3, 'learning_rate': 0.04089285700048085, 'n_estimators': 132, 'subsample': 0.6739417822102108, 'colsample_bytree': 0.9878338511058234, 'reg_alpha': 0.027189474714697306, 'reg_lambda': 2.8542399074977594, 'scale_pos_weight': 36.319868014475944}. Best is trial 1 with value: 0.37771803062023845.


...

[I 2026-09-01 14:55:16,131] Trial 90 finished with value: 0.3884392843875984 and parameters: {'max_depth': 4, 'min_child_weight': 3, 'learning_rate': 0.0855034835654475, 'n_estimators': 159, 'subsample': 0.937634315870018, 'colsample_bytree': 0.9723820885695529, 'reg_alpha': 1.264658685086037, 'reg_lambda': 7.138954057215735, 'scale_pos_weight': 32.95385395118808}. Best is trial 63 with value: 0.43197188542792897.
[I 2026-09-01 14:55:16,209] Trial 91 finished with value: 0.3975060305696 and parameters: {'max_depth': 4, 'min_child_weight': 5, 'learning_rate': 0.09473315737675951, 'n_estimators': 187, 'subsample': 0.9486705295454728, 'colsample_bytree': 0.8960363295240185, 'reg_alpha': 0.2521992228717454, 'reg_lambda': 0.392837422439358, 'scale_pos_weight': 38.31049282379126}. Best is trial 63 with value: 0.43197188542792897.
[I 2026-09-01 14:55:16,271] Trial 92 finished with value: 0.3894975463871172 and parameters: {'max_depth': 4, 'min_child_weight': 6, 'learning_rate': 0.125248203556

[I 2026-09-01 14:55:16,360] Trial 93 finished with value: 0.3303024466028973 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.019582973454466218, 'n_estimators': 183, 'subsample': 0.922723445295755, 'colsample_bytree': 0.8186808142041506, 'reg_alpha': 0.4273614642178547, 'reg_lambda': 0.022265115357803112, 'scale_pos_weight': 16.275765569710085}. Best is trial 63 with value: 0.43197188542792897.
[I 2026-09-01 14:55:16,436] Trial 94 finished with value: 0.3949233563639326 and parameters: {'max_depth': 4, 'min_child_weight': 5, 'learning_rate': 0.13743371425351453, 'n_estimators': 189, 'subsample': 0.9463446178796051, 'colsample_bytree': 0.8288160229184787, 'reg_alpha': 0.024853168765042115, 'reg_lambda': 0.11835072845466495, 'scale_pos_weight': 39.50995344966722}. Best is trial 63 with value: 0.43197188542792897.
[I 2026-09-01 14:55:16,522] Trial 95 finished with value: 0.38333203092299545 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate':

[I 2026-09-01 14:55:16,597] Trial 96 finished with value: 0.37481318532792135 and parameters: {'max_depth': 4, 'min_child_weight': 7, 'learning_rate': 0.1034054284483926, 'n_estimators': 172, 'subsample': 0.758448265794991, 'colsample_bytree': 0.8796873805032948, 'reg_alpha': 0.5314849688509389, 'reg_lambda': 0.048696405005825186, 'scale_pos_weight': 35.774447983650056}. Best is trial 63 with value: 0.43197188542792897.
[I 2026-09-01 14:55:16,682] Trial 97 finished with value: 0.36309761904258603 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.07165421344340558, 'n_estimators': 178, 'subsample': 0.9904677829327091, 'colsample_bytree': 0.8139845714848345, 'reg_alpha': 1.468183403115695e-08, 'reg_lambda': 0.5691274001994231, 'scale_pos_weight': 36.98981250024391}. Best is trial 63 with value: 0.43197188542792897.
[I 2026-09-01 14:55:16,771] Trial 98 finished with value: 0.37071665097462 and parameters: {'max_depth': 4, 'min_child_weight': 3, 'learning_rate': 0.

[I 2026-09-01 14:55:16,857] Trial 99 finished with value: 0.389996860177351 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.08072521713202292, 'n_estimators': 169, 'subsample': 0.9303991770295853, 'colsample_bytree': 0.8502452456205296, 'reg_alpha': 0.8020602168316183, 'reg_lambda': 0.05754271789084062, 'scale_pos_weight': 38.13578352099722}. Best is trial 63 with value: 0.43197188542792897.

Best parameters: {'max_depth': 4, 'min_child_weight': 1, 'learning_rate': 0.11283178173115917, 'n_estimators': 193, 'subsample': 0.9489421845738091, 'colsample_bytree': 0.7844495417927725, 'reg_alpha': 0.0001403754705562716, 'reg_lambda': 1.904572375721559, 'scale_pos_weight': 38.68235108390247}
Best cross-validation PR-AUC: 0.432 (baseline: 0.038)

Optimal threshold: 0.45
Best OOF F1: 0.424



Tuned Model 2 validation metrics:
model: Model 2 (generalization/scoring, target=collapse_onset_confirmed, event+near-miss population)
n_training_cities: 84
features: price_to_income_lag, price_to_income_5yr_chg, zhvi_yoy_lag, zhvi_qoq_lag, three-year_home_price_growth_trend, hpi_yoy_lag, hpi_3yr_chg_lag, pop_velocity_lag, pop_acceleration_lag, zori_yoy_lag, unemployment_rate_lag, inv_qoq_lag, sp500_yoy_lag, qcew_wage_yoy_lag, qcew_emp_yoy_lag
threshold: 0.45
accuracy: 0.9574468085106383
precision: 0.3684210526315789
recall: 0.7777777777777778
f1: 0.5
roc_auc: 0.9298611111111111
pr_auc: 0.44492948831184126
pr_auc_baseline: 0.02735562310030395

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98       320
           1       0.37      0.78      0.50         9

    accuracy                           0.96       329
   macro avg       0.68      0.87      0.74       329
weighted avg       0.98      0.96      0.96       32

In [17]:
model2_metrics = pd.DataFrame([tuned_metrics])
model2_metrics.to_csv("output/tables/model2_metrics_final.csv", index=False)
print("Saved output/tables/model2_metrics_final.csv")

print("\nConfirmed Model 1/Model 2 population identity:",
      model1_pool.sort_values(['cbsa','year','qtr']).reset_index(drop=True).equals(
          model2_pool.sort_values(['cbsa','year','qtr']).reset_index(drop=True)))

Saved output/tables/model2_metrics_final.csv

Confirmed Model 1/Model 2 population identity: True


## The watchlist: turning model output into something actionable

The model is trained on an enriched population -- only metros with a confirmed
collapse, plus near-miss metros -- because confirmed onsets among at-risk metros
run at roughly 1%, too sparse to learn from. That enrichment is the right
modeling choice and the wrong scoring assumption.

Left uncorrected the consequence was severe: the tuned operating threshold sat
at 0.31 while the highest-scoring metro in the deployment population scored
0.16, so **the alert could never fire on any metro**. The scores were being read
on the wrong scale entirely.

Two outputs are produced below, and the distinction is the point:

* **Rank, percentile, and tier** -- relative standing among the scored metros.
  Always valid, because rank ordering is exactly what the grouped
  cross-validation measured. This is the primary product.
* **`risk_probability`** -- the raw score shifted from the training prior onto
  the deployment prior. Interpretable as a probability, and anchored to a base
  rate measured from observed data rather than assumed.

In [18]:
# Both base rates are measured, not assumed: the deployment rate comes from
# every at-risk metro-quarter with complete features, not just the training set.
at_risk_all = df[~df["prev_unaffordable"]].dropna(
    subset=ALL_FEATURES + ["collapse_onset_confirmed"]
)
rates = base_rates(model2_pool, at_risk_all)
print("Base rates:", rates.describe())

holdout_scoring = pd.read_csv("output/holdout_scoring.csv")
raw_scores = model2_final.predict_proba(holdout_scoring[ALL_FEATURES])[:, 1]

city_risk = build_watchlist(holdout_scoring, raw_scores, rates)
city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)

print()
print(watchlist_summary(city_risk, top_n=15))

Base rates: training 3.574% vs deployment 0.874% (4.1x enriched)

Watchlist: 262 metros ranked by early-warning risk.

Read this as relative standing, not an absolute probability of collapse.
Base rates: training 3.574% vs deployment 0.874% (4.1x enriched); risk_probability is prior-corrected onto the deployment rate.

  Elevated    14 metros
  Watch       39 metros
  Monitor    209 metros

Top 15:
 risk_rank                        metro_name risk_tier  risk_score_raw  risk_probability
         1                      Kingston, NY  Elevated        0.103095          0.026619
         2                    Pittsfield, MA  Elevated        0.072912          0.018368
         3                     El Centro, CA  Elevated        0.056132          0.013952
         4                          Waco, TX  Elevated        0.032902          0.008029
         5                Killeen-Temple, TX  Elevated        0.030357          0.007394
         6 Grand Rapids-Wyoming-Kentwood, MI  Elevated        0.

In [19]:
top15 = city_risk.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
colors = {"Elevated": "#A6342B", "Watch": "#9C6B15", "Monitor": "#5C6876"}
ax.barh(
    top15["metro_name"],
    top15["risk_probability"],
    color=[colors[t] for t in top15["risk_tier"]],
)
ax.set_xlabel("Prior-corrected probability of confirmed onset")
ax.set_title("Top 15 metros by early-warning risk")

# The deployment base rate is the honest reference line: anything at this level
# is simply average for an at-risk metro.
ax.axvline(rates.deployment, ls="--", lw=1, color="#15202C")
ax.text(
    rates.deployment, -0.8,
    f"  base rate {rates.deployment:.2%}",
    va="center", fontsize=8, color="#15202C",
)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), title="Tier", loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig("output/figures/top15_highest_risk_metros.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved output/figures/top15_highest_risk_metros.png")

Saved output/figures/top15_highest_risk_metros.png


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_66808/1257217087.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Does the correction land where it should?

A prior correction is only as good as its anchor, so two checks below.

**Ranking must be unchanged.** Correction rescales every score by the same
monotonic transform, so the watchlist order has to be identical. If it is not,
something is wrong.

**The mean should sit below the anchor, not on it.** This is the part worth
reading carefully. The base rate is measured across *all* at-risk metros, but
the holdout excludes every training metro -- and training metros are precisely
the ones that had a confirmed onset. The scored population is therefore depleted
of events by construction, and its mean corrected probability landing under the
population-wide rate is the expected result rather than a calibration failure.

The corrected number answers: *if this metro were drawn from the general at-risk
population, how likely is a confirmed onset?* It is not a claim about a specific
metro's next quarter, and it should not be read as one.

In [20]:
mean_corrected = city_risk["risk_probability"].mean()
print(f"mean corrected probability   {mean_corrected:.3%}")
print(f"observed deployment rate     {rates.deployment:.3%}")
print(f"mean raw (training-scale)    {city_risk['risk_score_raw'].mean():.3%}")

# Correction must not reshuffle the watchlist -- ranking is what CV validated.
by_raw = city_risk.sort_values("risk_score_raw", ascending=False)["cbsa"].tolist()
by_prob = city_risk.sort_values("risk_probability", ascending=False)["cbsa"].tolist()
print(f"\nranking preserved by correction: {by_raw == by_prob}")

print("\nTier distribution:")
print(city_risk["risk_tier"].value_counts().to_string())

mean corrected probability   0.073%
observed deployment rate     0.874%
mean raw (training-scale)    0.300%

ranking preserved by correction: True

Tier distribution:
risk_tier
Monitor     209
Watch        39
Elevated     14


## Validation testing

Grouped cross-validation (`StratifiedGroupKFold`, cities never split across folds) across the full training population.

In [21]:
n_splits_m1 = min(10, groups_m1.nunique())
group_cv = StratifiedGroupKFold(n_splits=n_splits_m1, shuffle=True, random_state=42)

cv_model1 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=scale_pos_weight_1,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
auc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")
prauc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="average_precision")

print(f"Model 1: grouped cross-validation ({n_splits_m1} folds)")
print(f"Mean AUC: {np.nanmean(auc_scores_m1):.3f} (std {np.nanstd(auc_scores_m1):.3f})")
print(f"Mean PR-AUC: {np.nanmean(prauc_scores_m1):.3f} (std {np.nanstd(prauc_scores_m1):.3f}) "
      f"-- baseline (positive rate): {y_all.mean():.3f}")

n_splits_m2 = min(10, groups_m2.nunique())
group_cv_m2 = StratifiedGroupKFold(n_splits=n_splits_m2, shuffle=True, random_state=42)
cv_model2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=class_ratio_2,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
auc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")
prauc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision")

print(f"\nModel 2: grouped cross-validation ({n_splits_m2} folds)")
print(f"Mean AUC: {np.nanmean(auc_scores_m2):.3f} (std {np.nanstd(auc_scores_m2):.3f})")
print(f"Mean PR-AUC: {np.nanmean(prauc_scores_m2):.3f} (std {np.nanstd(prauc_scores_m2):.3f}) "
      f"-- baseline (positive rate): {yb_all.mean():.3f}")

cv_results = pd.DataFrame({
    "model": ["model 1"] * len(auc_scores_m1) + ["model 2"] * len(auc_scores_m2),
    "fold": list(range(1, len(auc_scores_m1) + 1)) + list(range(1, len(auc_scores_m2) + 1)),
    "roc_auc": list(auc_scores_m1) + list(auc_scores_m2),
    "pr_auc": list(prauc_scores_m1) + list(prauc_scores_m2),
})
cv_results.to_csv("output/tables/cv_results_final.csv", index=False)
print("\nSaved output/tables/cv_results_final.csv")

Model 1: grouped cross-validation (10 folds)
Mean AUC: 0.915 (std 0.042)
Mean PR-AUC: 0.370 (std 0.199) -- baseline (positive rate): 0.036



Model 2: grouped cross-validation (10 folds)
Mean AUC: 0.915 (std 0.042)
Mean PR-AUC: 0.370 (std 0.199) -- baseline (positive rate): 0.036

Saved output/tables/cv_results_final.csv


## Model comparison: logistic regression baseline

In [22]:
logit_baseline_m1 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
prauc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="average_precision")
auc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")

logit_baseline_m2 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
prauc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision")
auc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")

comparison = pd.DataFrame([
    {"model": "Model 1 XGBoost", "mean_pr_auc": np.nanmean(prauc_scores_m1), "mean_auc": np.nanmean(auc_scores_m1)},
    {"model": "Model 1 Logistic Regression", "mean_pr_auc": np.nanmean(prauc_lr_m1), "mean_auc": np.nanmean(auc_lr_m1)},
    {"model": "Model 2 XGBoost", "mean_pr_auc": np.nanmean(prauc_scores_m2), "mean_auc": np.nanmean(auc_scores_m2)},
    {"model": "Model 2 Logistic Regression", "mean_pr_auc": np.nanmean(prauc_lr_m2), "mean_auc": np.nanmean(auc_lr_m2)},
])
comparison.to_csv("output/tables/model_comparison_logreg_vs_xgboost.csv", index=False)
print(comparison)

                         model  mean_pr_auc  mean_auc
0              Model 1 XGBoost     0.369951  0.914597
1  Model 1 Logistic Regression     0.307339  0.899187
2              Model 2 XGBoost     0.369951  0.914597
3  Model 2 Logistic Regression     0.307339  0.899187


## SHAP explainability (diagnostic: with vs. without the price-to-income level features)

In [23]:
model2_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    scale_pos_weight=class_ratio_2,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
model2_full.fit(Xb_all, yb_all)

explainer2 = shap.TreeExplainer(model2_full)
shap_values2 = explainer2.shap_values(Xb_all)

plt.figure()
shap.summary_plot(shap_values2, Xb_all, plot_type="bar", show=False)
plt.title("Model 2 -- all features, target=collapse_onset_confirmed")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values2, Xb_all, show=False)
plt.title("Model 2 -- all features, target=collapse_onset_confirmed")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_beeswarm_full.png", dpi=150)
plt.close()

shap_importance_full_m2 = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values2).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance_full_m2.to_csv("output/tables/shap_importance_model2_full.csv", index=False)
print("Model 2 (all features) SHAP importance:")
print(shap_importance_full_m2)

Model 2 (all features) SHAP importance:
                               feature  mean_abs_shap
0                  price_to_income_lag       1.105351
3                         zhvi_qoq_lag       0.736196
10               unemployment_rate_lag       0.441837
4   three-year_home_price_growth_trend       0.406296
12                       sp500_yoy_lag       0.400378
14                    qcew_emp_yoy_lag       0.285083
8                 pop_acceleration_lag       0.215820
11                         inv_qoq_lag       0.211670
9                         zori_yoy_lag       0.177643
13                   qcew_wage_yoy_lag       0.134714
2                         zhvi_yoy_lag       0.114107
5                          hpi_yoy_lag       0.073472
6                      hpi_3yr_chg_lag       0.038817
7                     pop_velocity_lag       0.018701
1              price_to_income_5yr_chg       0.010405


In [24]:
NO_LEVEL_FEATURES = [f for f in ALL_FEATURES if f not in ("price_to_income_lag", "price_to_income_5yr_chg")]

cv_model2_noleveL = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=class_ratio_2, random_state=42
)
auc_noleveL = cross_val_score(
    cv_model2_noleveL, Xb_all[NO_LEVEL_FEATURES], yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc"
)
prauc_noleveL = cross_val_score(
    cv_model2_noleveL, Xb_all[NO_LEVEL_FEATURES], yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision"
)

diagnostic = pd.DataFrame([
    {"feature_set": "All 15 features", "mean_auc": np.nanmean(auc_scores_m2), "mean_pr_auc": np.nanmean(prauc_scores_m2)},
    {"feature_set": "Without price-to-income level features (13 left)", "mean_auc": np.nanmean(auc_noleveL), "mean_pr_auc": np.nanmean(prauc_noleveL)},
])
diagnostic.to_csv("output/tables/diagnostic_without_level_features.csv", index=False)
print(diagnostic)
print(f"\nBaseline PR-AUC (positive rate): {yb_all.mean():.3f}")

                                        feature_set  mean_auc  mean_pr_auc
0                                   All 15 features  0.914597     0.369951
1  Without price-to-income level features (13 left)  0.848939     0.312935

Baseline PR-AUC (positive rate): 0.036


## Backtesting

Two backtests, answering different questions. The distinction matters, because
they disagree.

**Cross-sectional (leave-one-city-out)** — below — asks *does this generalize to
a metro it has never seen?* It is a fair test of that, but it trains on the whole
time period, so a model predicting a 2020 event may have learned from 2023 data.

**Walk-forward (temporal)** — the section after — asks the question an
early-warning system is actually judged on: *if we had been running this in 2021,
what would it have told us?* At each origin quarter the model is refit using only
labels observable at that point, then scores every at-risk metro for that
quarter. Nothing after the origin is visible.

In [25]:
def leave_one_city_out_backtest(X_all_bt, y_all_bt, groups, label):
    aucs = {}
    for city_code in sorted(groups.unique()):
        train_mask = groups != city_code
        test_mask = groups == city_code
        y_train_bt, y_test_bt = y_all_bt[train_mask], y_all_bt[test_mask]

        if y_train_bt.nunique() < 2 or y_test_bt.nunique() < 2:
            continue

        X_train_bt = X_all_bt[train_mask].copy()
        X_train_bt.insert(0, "const", 1.0)
        X_test_bt = X_all_bt[test_mask].copy()
        X_test_bt.insert(0, "const", 1.0)

        result = Logit(y_train_bt, X_train_bt).fit_regularized(method="l1", alpha=1.0, disp=0)
        pred_probs = result.predict(X_test_bt)
        auc = roc_auc_score(y_test_bt, pred_probs)
        aucs[city_code] = auc

    if aucs:
        vals = np.array(list(aucs.values()))
        print(f"{label}: {len(aucs)} cities had both classes present in their held-out fold")
        print(f"  mean AUC: {vals.mean():.3f}, median: {np.median(vals):.3f}, "
              f"min: {vals.min():.3f}, max: {vals.max():.3f}")
        worst = sorted(aucs.items(), key=lambda kv: kv[1])[:5]
        best = sorted(aucs.items(), key=lambda kv: -kv[1])[:5]
        city_names = model2_pool.groupby("cbsa")["metro_name"].first()
        print("  worst 5:", [(city_names.get(c, c), round(a, 3)) for c, a in worst])
        print("  best 5:", [(city_names.get(c, c), round(a, 3)) for c, a in best])
    return aucs


print("Model 2 backtest")
auc2_by_city = leave_one_city_out_backtest(Xb_all, yb_all, groups_m2, "Model 2")

Model 2 backtest


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


...

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


Model 2: 52 cities had both classes present in their held-out fold
  mean AUC: 0.874, median: 0.933, min: 0.000, max: 1.000
  worst 5: [('College Station-Bryan, TX', 0.0), ('Springfield, MA', 0.167), ('Lake Havasu City-Kingman, AZ', 0.4), ('Wilmington, NC', 0.429), ('Salisbury, MD', 0.6)]
  best 5: [('Austin-Round Rock-San Marcos, TX', 1.0), ('Bridgeport-Stamford-Danbury, CT', 1.0), ('Charleston-North Charleston, SC', 1.0), ('Charlotte-Concord-Gastonia, NC-SC', 1.0), ('Crestview-Fort Walton Beach-Destin, FL', 1.0)]


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


## Walk-forward backtest

This is the honest test of the early-warning claim, and the number to quote.

In [26]:
bt = walk_forward(df, start=(2021, 1))
print(bt.summary())

print("\n\nPer-origin (origins containing at least one actual onset)")
print("-" * 78)
cols = ["origin", "n_train_events", "n_scored", "n_events",
        "pr_auc", "precision_at_10", "recall_at_10", "recall_at_20"]
print(bt.evaluable[cols].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

bt.predictions.to_csv("output/tables/backtest_predictions.csv", index=False)
bt.per_origin.to_csv("output/tables/backtest_per_origin.csv", index=False)

Walk-forward backtest
----------------------------------------------------------
  origins evaluated      8
  metro-quarters scored  4,511
  actual onsets          43
  base rate              0.953%

  pooled PR-AUC          0.053   (no-skill 0.010)
  pooled ROC-AUC         0.866

  mean precision@10      0.275
  mean recall@10         0.533
  mean recall@20         0.840
  median event rank      10  (of ~200 scored)


Per-origin (origins containing at least one actual onset)
------------------------------------------------------------------------------
origin  n_train_events  n_scored  n_events  pr_auc  precision_at_10  recall_at_10  recall_at_20
2021Q1               9       210         8   0.668            0.600         0.750         0.750
2021Q2              17       203         3   0.221            0.200         0.667         1.000
2021Q3              20       200         9   0.400            0.300         0.333         1.000
2021Q4              29       190         3   0.137      

### Lead time: how early does it flag them?

For every metro that actually had a confirmed onset, this traces the rank it held
in the quarters *before* it happened. Percentile is reported alongside rank
because the number of scored metros varies by quarter.

Read the longer horizons with care: the backtest window starts in 2021, so a
metro that collapsed early in the window simply cannot be traced back eight
quarters. Those rows are both few and biased toward late-collapsing metros.

In [27]:
lead = lead_time(bt, horizon=8)
summary = lead_time_summary(lead)
summary.to_csv("output/tables/backtest_lead_time.csv", index=False)

print("Rank held before onset (0 = the onset quarter itself)")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

pooled = bt.pooled()
print("\n\nThe two backtests, same model:")
print("-" * 66)
print(f"  cross-sectional (grouped CV)   PR-AUC {np.nanmean(prauc_scores_m2):.3f}"
      f"  baseline {yb_all.mean():.3f}   {np.nanmean(prauc_scores_m2)/yb_all.mean():.1f}x")
print(f"  walk-forward (temporal)        PR-AUC {pooled['pr_auc']:.3f}"
      f"  baseline {pooled['base_rate']:.3f}   {pooled['pr_auc']/pooled['base_rate']:.1f}x")
print("\nThe temporal number is the one to quote. Cross-sectional validation is")
print("roughly twice as optimistic here, which is what a backtest is for.")

Rank held before onset (0 = the onset quarter itself)
 quarters_before_onset  events  median_rank  median_percentile  pct_in_top_10  pct_in_top_20
                     0      43       10.000              0.951          0.512          0.791
                     1      34       13.000              0.939          0.324          0.706
                     2      32       23.000              0.890          0.188          0.438
                     3      23       39.000              0.810          0.261          0.304
                     4      20       64.000              0.700          0.100          0.300
                     5      12       60.500              0.717          0.083          0.083
                     6       2       11.500              0.943          0.500          1.000
                     7       2       22.500              0.885          0.000          0.500
                     8       2       29.000              0.853          0.000          0.000


The two backte